# LightGBM Fraud Detection Model

This notebook trains a LightGBM classifier using the engineered fraud indicators developed during exploratory analysis.

The objective is to evaluate whether gradient boosting with histogram-based tree construction can improve fraud detection performance compared to the CatBoost baseline.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/DataSet.csv")

In [ ]:
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

## Feature Engineering

Several fraud indicators and interaction features were created based on patterns discovered during exploratory analysis.

These features capture high-risk behavioural patterns and missing-value signals associated with fraudulent accounts.

In [ ]:
df["F115_high"] = (df["F115"] > 0.78).astype(int)

df["F2582_hot"] = (
    (df["F2582"] > -0.03)
    & (df["F2582"] <= 0)
).astype(int)

df["F2956_hot"] = (
    (df["F2956"] > 19)
    & (df["F2956"] <= 32)
).astype(int)

df["F531_hot"] = (
    (df["F531"] > 0.95)
    & (df["F531"] <= 1.35)
).astype(int)

df["F2737_safe"] = (
    (df["F2737"] > 0)
    & (df["F2737"] <= 0.04)
).astype(int)

df["F2582_F531"] = (
    df["F2582_hot"]
    & df["F531_hot"]
).astype(int)

df["fraud_cluster_1"] = (
    df["F2582_hot"]
    & df["F2956_hot"]
    & df["F531_hot"]
).astype(int)

df["f3836_hot"] = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
).astype(int)

df["F2956_F115"] = (
    df["F2956_hot"] &
    df["F115_high"]
).astype(int)

df["F2582_pos_F2956_low"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

## Feature Selection

The final feature set combines original variables, engineered fraud indicators, interaction features, and missing-value features.

These features were selected through iterative experimentation and fraud-rate analysis.

In [ ]:
features = [
    "F115",
    "F2582",
    "F2956",
    "F531",
    "F2737",
    "F670",
    "F673",
    "F3891",
    "F3889",

    # engineered features

    "F115_high",
    "F2582_hot",
    "F2956_hot",
    "F531_hot",
    "F2737_safe",
    "F2582_F531",
    "fraud_cluster_1",
    "f3836_hot",
    "F2956_F115",
    "F2582_pos_F2956_low"

]

family_cols = [
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675",

    "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

In [ ]:
X = pd.get_dummies(
    df[features],
    columns=["F3891", "F3889"],
    dummy_na=True
)

y = df["F3924"]

## Data Preparation

Categorical variables are encoded using one-hot encoding and a stratified train-test split is applied to preserve fraud prevalence across datasets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

## Model Training

A LightGBM classifier is trained using class weighting to address the severe imbalance between legitimate and fraudulent accounts.

Model performance is evaluated primarily through fraud recall and Average Precision.

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm.fit(X_train, y_train)

## Threshold Optimization

Different probability thresholds are evaluated to understand the precision-recall tradeoff and identify operating points suitable for fraud detection.

In [ ]:
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

In [ ]:
for threshold in [0.5, 0.4, 0.3, 0.25, 0.2]:

    y_pred = (y_prob_lgbm >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")

    print(confusion_matrix(y_test, y_pred))

    print(classification_report(y_test, y_pred))

## Ranking Performance

Average Precision (AP) measures the model's ability to rank fraudulent accounts above legitimate accounts across all thresholds.

In [ ]:
ap = average_precision_score(y_test, y_prob_lgbm)

print("AP:", ap)

## Prediction Export

Predicted probabilities are stored for later comparison with XGBoost and Neural Network models during ensemble construction.

In [ ]:
lgbm_probs = pd.DataFrame({
    "prob": y_prob_lgbm
}, index=X_test.index)

lgbm_probs.to_csv("../data/processed/lgbm_probs.csv")

## Key Findings

The LightGBM model achieved performance comparable to XGBoost and validated the predictive value of the engineered fraud indicators.

Important observations:

- Missing-value indicators contributed significantly to model performance.
- Fraud detection improved substantially compared to the baseline CatBoost model.
- The model demonstrated strong ranking performance as measured by Average Precision.
- Results were later incorporated into an ensemble framework for further evaluation.